# Capstone: Mitigation study: one expectation value

**Quantum Computing Advanced · Module 6 · about 5 hours** · notebook version 2026-10-09

**The question of this experiment.** How close to the ideal value can you bring one noisy expectation value, ⟨X X X⟩ of a three-qubit GHZ state, with readout mitigation and zero-noise extrapolation, how certain is each mitigated result, and when is mitigation worth its cost in shots?

You run the experiment on a **noisy simulator**, a simulator that adds the errors of a real quantum computer described by a **noise model**. You compare what you measure with what the noise model predicts, say how certain every result is, and say what the experiment cannot tell you. The capstone brings together the code, the mitigation and the application of Modules 2, 3 and 4, and you write more of the code yourself than in a lab.

**The plan.**

1. write the statistics functions every capstone uses (Step 1) and meet the capstone noise model (Step 2);
2. build the GHZ circuit and measure **⟨X X X⟩** from shots (Step 3);
3. remove the **readout error** by inverting the readout confusion (Step 4);
4. amplify the gate noise by **global folding** (Step 5);
5. extrapolate to zero noise with **weights** you compute (Step 6);
6. measure the uncertainty of every result with the **bootstrap** and decide which method is best for a given **shot budget** (Step 7);
7. repeat the study for 2 to 5 qubits (Step 8);
8. get your verification value (Step 9).

**How it is graded.** Your capstone quiz in Canvas (the one for this experiment, 20 points, pass at 16) asks for your verification value from the check cell (6 points), three results from this notebook and four questions about uncertainty, interpretation and limits (2 points each). Question 1 gives you three personal numbers: NQ, P2 and READOUT. At the end you write a short **summary** of your experiment (required, not graded), which you submit in the Course Completion module.

**Saving your work.** JupyterLite keeps your changes in this browser, so you can stop and come back. To keep a copy elsewhere, use **File → Download**. Run each code cell with **Shift + Enter**, in order; if you come back later, run the cells from the top again.

## Step 0: start Python

Run the cell below. The first time, Python can take up to a minute to start in your browser, and Matplotlib takes a few more seconds to load.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import qsim
from qsim import QuantumCircuit, NoiseModel, ReadoutError, depolarizing_error, simulate_density_matrix

print("Ready. qsim", qsim.__version__, "| NumPy", np.__version__)

## Step 1: how certain is a result from shots?

Every number you measure in this capstone comes from shots, so every number has an uncertainty. Three formulas cover all of them.

**A fraction.** If a result has probability p and you take S shots, the fraction f of shots with that result has standard deviation

**σ = √( p (1 − p) / S )**.

**An expectation value of ±1 results**, such as ⟨Z⟩ or ⟨X X X⟩: each shot gives +1 or −1, and the average E has standard deviation

**σ = √( (1 − E²) / S )**.

This is twice the σ of the fraction of +1 results, because E = 2 f − 1. For E = 0.8 and S = 4,000 it is √(0.36 / 4000) = 0.0095.

**A combination of independent results.** Mitigation combines several measured values: an estimate Ê = c₁ E₁ + c₂ E₂ + ... from independent runs with uncertainties σ₁, σ₂, ... has

**σ = √( (c₁ σ₁)² + (c₂ σ₂)² + ... )**.

Uncertainties add **in quadrature**. The difference of two results is the special case c = (1, −1): σ_diff = √(σ₁² + σ₂²). Large coefficients, as in an extrapolation, make the combination much less certain than any single run: this is the price of mitigation you met in Module 3.

**The course rule: 3σ.** A measured value agrees with an expected value when they differ by at most 3σ. A difference larger than 3σ_diff is **significant**: shot noise alone would almost never produce it. A smaller difference may be real, but your data cannot show it. Two cautions: the rule treats the spread as normal (bell-shaped), which is good when many shots give each result; and when you compare many settings, chance alone makes one "significant" difference likely somewhere (with 20 comparisons, about 5%). Treat a single surprising difference among many as a question to test again with new shots.

**Your task:** write the three functions below. Every later step uses them.

- `fraction_sigma(p, shots)`: √(p (1 − p) / shots).
- `expectation_sigma(E, shots)`: √((1 − E²) / shots).
- `combined_sigma(coefficients, sigmas)`: √(Σ (cᵢ σᵢ)²) for two lists of the same length.

In [ ]:
def fraction_sigma(p, shots):
    """The standard deviation of a fraction measured with `shots` shots, when the true probability is p."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete fraction_sigma() first.")


def expectation_sigma(E, shots):
    """The standard deviation of the average of `shots` results of +1 or -1 with expectation value E."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete expectation_sigma() first.")


def combined_sigma(coefficients, sigmas):
    """The standard deviation of sum(c_i * E_i) for independent E_i with standard deviations sigma_i."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete combined_sigma() first.")

In [ ]:
print("fraction_sigma(0.0613, 4000) =", round(fraction_sigma(0.0613, 4000), 5), "  (expected 0.00379)")
print("expectation_sigma(0.8, 4000) =", round(expectation_sigma(0.8, 4000), 5), "  (expected 0.00949)")
print("combined_sigma([1, -1], [0.01, 0.02]) =", round(combined_sigma([1, -1], [0.01, 0.02]), 5), "  (expected 0.02236)")
print("combined_sigma([1.875, -1.25, 0.375], [0.012, 0.015, 0.02]) =",
      round(combined_sigma([1.875, -1.25, 0.375], [0.012, 0.015, 0.02]), 5), "  (expected 0.03023)")

## Step 2: the capstone noise model

The noise model of the capstone is the one you used in Level 2 Module 6, with one addition:

- after every **two-qubit gate**, a **depolarizing error** with parameter P2 (with probability P2 the two qubits are replaced by a completely random state);
- after every **one-qubit gate**, the same on one qubit with P2 / 10;
- an **idle error** on `id` gates, used only by the repetition-code capstone (a qubit that waits while others are measured);
- at **readout**, a 0 is recorded as 1 with probability READOUT and a 1 as 0 with probability 2 · READOUT.

The course values are **P2 = 0.01** and **READOUT = 0.02**, several times worse than IBM's best devices, so that the effects show clearly in a few thousand shots. Your capstone quiz gives you your own P2 and READOUT (0.005 to 0.030 each) for the check cell.

The prepared cell builds the model and three tools:

- `exact_probabilities(qc, p2, readout)`: the noise model's **exact** probability of every recorded result, from the density matrix (Module 1) and the readout confusion; no shots, no randomness. This is the expected value the 3σ rule compares with.
- `sample_counts(probs, shots, seed)`: random shots drawn from those probabilities. A run of the noisy simulator gives counts with exactly these statistics; drawing them from the exact probabilities is much faster in the browser, and the seed makes your run repeatable.
- `apply_readout(probs, readout)`: the readout errors alone, applied to probabilities.

In [ ]:
ONE_QUBIT_GATES = ["h", "x", "y", "z", "rx", "ry", "rz", "p", "s", "sdg", "t", "tdg", "sx"]
TWO_QUBIT_GATES = ["cx", "cz", "cp", "rzz", "swap"]
P2_COURSE, READOUT_COURSE = 0.01, 0.02


def confusion(readout):
    """Prepared. Rows: the true bit (0, 1); columns: the recorded bit."""
    return np.array([[1 - readout, readout], [2 * readout, 1 - 2 * readout]])


def capstone_noise_model(p2=P2_COURSE, readout=READOUT_COURSE, idle=0.0):
    """Prepared: depolarizing error p2 after two-qubit gates, p2/10 after one-qubit gates, idle on id gates, readout."""
    nm = NoiseModel()
    if p2 > 0:
        nm.add_all_qubit_quantum_error(depolarizing_error(p2 / 10, 1), ONE_QUBIT_GATES)
        nm.add_all_qubit_quantum_error(depolarizing_error(p2, 2), TWO_QUBIT_GATES)
    if idle > 0:
        nm.add_all_qubit_quantum_error(depolarizing_error(idle, 1), ["id"])
    if readout > 0:
        nm.add_all_qubit_readout_error(ReadoutError(confusion(readout).tolist()))
    return nm


def apply_readout(probs, readout):
    """Prepared. probs over m recorded bits (index bit j = classical bit j); each bit is flipped by the confusion."""
    probs = np.asarray(probs, dtype=float)
    m = int(round(math.log2(len(probs))))
    c = confusion(readout)
    for j in range(m):
        t = probs.reshape(2 ** (m - 1 - j), 2, 2 ** j)
        probs = np.einsum("aib,ij->ajb", t, c).reshape(-1)
    return probs


def exact_probabilities(qc, p2=P2_COURSE, readout=READOUT_COURSE, idle=0.0):
    """Prepared: the noise model's exact probability of each recorded result, {"bits": p}, Qiskit's key order.
    qc must end with its measurements (each classical bit receives one qubit). No shots are taken."""
    measured = {inst.clbits[0]: inst.qubits[0] for inst in qc.data if inst.name == "measure"}
    body = qc.copy()
    body.remove_final_measurements()
    nm = capstone_noise_model(p2, 0.0, idle)
    rho = simulate_density_matrix(body, nm if not nm.is_ideal() else None)   # the exact mixed state after the noisy gates
    m = len(measured)
    probs = rho.probabilities(qargs=[measured[j] for j in range(m)])
    probs = apply_readout(probs, readout)                                       # then the readout errors
    return {format(i, f"0{m}b"): float(p) for i, p in enumerate(probs) if p > 1e-15}


def sample_counts(probs, shots, seed):
    """Prepared: `shots` random results drawn from exact probabilities, as a quantum computer gives them."""
    keys = sorted(probs)
    p = np.array([probs[k] for k in keys])
    draw = np.random.default_rng(seed).multinomial(shots, p / p.sum())
    return {k: int(c) for k, c in zip(keys, draw) if c > 0}


print(capstone_noise_model())

## Step 3: the expectation value

The **GHZ state** (|000⟩ + |111⟩)/√2 of Level 2 is made by H on qubit 0 and a chain of CX gates. Its ⟨X X X⟩ is exactly **1**: the state is unchanged when all three qubits are flipped. Noise brings the measured value below 1, and this capstone tries to get it back.

A quantum computer measures in the Z basis. To measure X on every qubit, add an H gate on every qubit before the measurement (H turns X into Z). Then each shot gives a key of n bits, and the product of the X results is +1 for an **even** number of 1s and −1 for an **odd** number. So ⟨X X X⟩ is the **parity**:

⟨X X X⟩ = Σ over results of (−1)^(number of 1s) · (fraction of shots with that result).

**Your task:**

- `ghz_circuit(n)`: a QuantumCircuit on n qubits: H on qubit 0, CX(0, 1), CX(1, 2), ..., CX(n − 2, n − 1), then H on every qubit. **No measurements**: Step 5 folds this circuit (runs it, its inverse and it again), and a measurement cannot be inverted. The prepared `measured(qc)` adds them (qubit i into classical bit i).
- `parity_expectation(probs)`: the parity above, for a dictionary of counts or of probabilities. Divide by the total, so both work.

In [ ]:
def ghz_circuit(n):
    """H on qubit 0, the CX chain, then H on every qubit (to measure X); no measurements."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete ghz_circuit() first.")


def parity_expectation(probs):
    """<X...X>: sum of (-1)^(number of 1s) * weight, divided by the total weight."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete parity_expectation() first.")

In [ ]:
def measured(qc):
    """Prepared: a copy of qc with every qubit measured (qubit i into classical bit i)."""
    out = QuantumCircuit(qc.num_qubits, qc.num_qubits)
    out.compose(qc, inplace=True)
    for i in range(qc.num_qubits):
        out.measure(i, i)
    return out


NQ_DEMO, SHOTS = 3, 4000
print(measured(ghz_circuit(NQ_DEMO)).draw())
ideal = exact_probabilities(measured(ghz_circuit(NQ_DEMO)), 0, 0)
print("Without noise:", {k: round(v, 3) for k, v in ideal.items()}, " <XXX> =", round(parity_expectation(ideal), 4), "  (expected 1.0)")

exact1 = exact_probabilities(measured(ghz_circuit(NQ_DEMO)))
counts1 = sample_counts(exact1, SHOTS, seed=1)
E_exact, E_run = parity_expectation(exact1), parity_expectation(counts1)
s = expectation_sigma(E_exact, SHOTS)
print(f"With noise: exact <XXX> = {E_exact:.4f};  your run of {SHOTS} shots: {E_run:.4f} +- {s:.4f} ({(E_run - E_exact) / s:+.1f} sigma)")

**What to notice.** Without noise every shot has an even number of 1s and ⟨X X X⟩ = 1. With the course noise model the exact value is **0.8108**: about 19% below the ideal, far more than the shot noise σ ≈ 0.0093 of 4,000 shots. Shot noise is random and shrinks with more shots; this **bias** comes from the noise and stays however many shots you take. Mitigation attacks the bias.

## Step 4: readout mitigation

Part of the bias comes from reading out. In the capstone model, each bit is recorded wrongly with the confusion matrix C (rows: the true bit, columns: the recorded bit):

C = [[1 − r, r], [2r, 1 − 2r]] with r = READOUT.

For one bit, the measured probabilities are m[recorded] = Σ over true of C[true][recorded] · t[true], that is **m = Cᵀ t**, so the true probabilities are **t = (Cᵀ)⁻¹ m**. The errors of different bits are independent, so for n bits you apply the 2 × 2 inverse to each bit in turn (a tensor product of inverses, the method of Module 3). The prepared `apply_readout` does the forward direction (it applies Cᵀ bit by bit); your function undoes it.

Bit j of a key is the character n − 1 − j (qubit 0 is the rightmost). One way: put the probabilities in a vector v indexed by int(key, 2), and for each bit j reshape it to (2^(n−1−j), 2, 2^j) and apply the 2 × 2 matrix to the middle axis with `np.einsum("ij,ajb->aib", inverse, t)`, exactly as `apply_readout` does with C.

**Your task:** `mitigate_readout(probs, readout)`: a dictionary {key: corrected probability} with every key of n bits. Normalize the input first (it may be counts). The corrected values may be slightly negative when they come from shots: that is the statistical noise of an inverse, and the parity still works.

In [ ]:
def mitigate_readout(probs, readout):
    """Probabilities corrected for readout errors: (C^T)^-1 applied to every bit."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete mitigate_readout() first.")

In [ ]:
print("Readout-mitigated, exact:   <XXX> =", round(parity_expectation(mitigate_readout(exact1, READOUT_COURSE)), 4))
print("Readout-mitigated, your run: <XXX> =", round(parity_expectation(mitigate_readout(counts1, READOUT_COURSE)), 4))
check = mitigate_readout({k: float(v) for k, v in zip(["00", "01", "10", "11"], apply_readout([0.1, 0.2, 0.3, 0.4], 0.02))}, 0.02)
print("Undoing apply_readout:", {k: round(v, 6) for k, v in check.items()}, "  (expected 0.1, 0.2, 0.3, 0.4)")

**What to notice.** Readout mitigation moves the exact value from 0.8108 to **0.9762**: most of the bias came from reading out. What is left, about 0.024, comes from the gates, and no readout correction can remove it. The price: the corrected value is less certain (Step 7 measures by how much), because the inverse amplifies the shot noise of every bit.

## Step 5: amplify the noise by folding

**Zero-noise extrapolation** (ZNE, Module 3) measures the value at several noise levels and extrapolates back to zero noise. To raise the gate noise without changing what the circuit does, **global folding** runs the circuit U, then U⁻¹ U, (scale − 1)/2 times:

- scale 1: U;
- scale 3: U U⁻¹ U;
- scale 5: U U⁻¹ U U⁻¹ U.

U⁻¹ U does nothing ideally, but its gates make errors, so scale 3 has about three times the gate noise of scale 1. Folding the whole circuit (global) is the simplest choice; folding single gates (local) gives finer scales, as the spin-chain capstone does.

**Your task:** `fold_global(qc, scale)` for an odd scale: a **new** circuit (start from `qc.copy()`, do not change `qc`), with `qc.inverse()` and `qc` composed (scale − 1)/2 times. `out.compose(other, inplace=True)` appends a circuit.

In [ ]:
def fold_global(qc, scale):
    """qc followed by (scale - 1) / 2 pairs of (qc inverse, qc); scale odd. qc itself is not changed."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete fold_global() first.")

In [ ]:
SCALES = [1, 3, 5]
base = ghz_circuit(NQ_DEMO)
exact_s, raw_s, ro_s = {}, {}, {}
print(f"{'scale':>6s}{'gates':>7s}{'raw <XXX>':>11s}{'readout-mitigated':>19s}")
for s in SCALES:
    qc = fold_global(base, s)
    exact_s[s] = exact_probabilities(measured(qc))
    raw_s[s] = parity_expectation(exact_s[s])
    ro_s[s] = parity_expectation(mitigate_readout(exact_s[s], READOUT_COURSE))
    print(f"{s:6d}{len(qc.data):7d}{raw_s[s]:11.4f}{ro_s[s]:19.4f}")

**What to notice.** The readout-mitigated values fall with the scale (0.9762, 0.9302, 0.8865): folding added gate noise and nothing else, as intended. The raw values fall too, but they also carry the readout error, which folding does **not** amplify: the circuit is measured once at every scale.

## Step 6: extrapolate to zero noise

Fit a polynomial of degree d to the points (scale, value) by least squares, and read it at scale 0. The answer is a **weighted sum** of the measured values, Ê = Σ cᵢ Eᵢ, and the weights depend only on the scales:

- write the fit as V a ≈ E, where V is the **Vandermonde** matrix with rows (1, sᵢ, sᵢ², ..., sᵢ^d);
- the least-squares coefficients are a = V⁺ E, with V⁺ the pseudo-inverse (`np.linalg.pinv`);
- the value at scale 0 is a₀, the first coefficient, so the weights are **the first row of V⁺**.

With three scales, degree 1 is a least-squares line and degree 2 passes through all three points (**Richardson extrapolation**). Knowing the weights is what lets you compute the uncertainty of the result with `combined_sigma`.

**Your task:** `zne_coefficients(scales, degree)`: the list of weights, one per scale. `np.vander(np.asarray(scales, dtype=float), degree + 1, increasing=True)` builds V.

In [ ]:
def zne_coefficients(scales, degree):
    """Weights c_i with value(0) = sum c_i E_i for a least-squares polynomial of this degree."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete zne_coefficients() first.")

In [ ]:
lin, quad = zne_coefficients(SCALES, 1), zne_coefficients(SCALES, 2)
print("linear weights:   ", [round(float(c), 4) for c in lin], "  (expected [1.0833, 0.3333, -0.4167])")
print("quadratic weights:", [round(float(c), 4) for c in quad], "  (expected [1.875, -1.25, 0.375])")
zne = lambda c, vals: sum(ci * vals[s] for ci, s in zip(c, SCALES))
print(f"\nExact values (ideal 1):")
print(f"   raw, scale 1:                    {raw_s[1]:.4f}")
print(f"   readout-mitigated, scale 1:      {ro_s[1]:.4f}")
print(f"   ZNE linear, without readout fix: {zne(lin, raw_s):.4f}")
print(f"   readout + ZNE linear:            {zne(lin, ro_s):.4f}")
print(f"   readout + ZNE quadratic:         {zne(quad, ro_s):.4f}")

**What to notice.** Each weight list adds up to 1 (a value that does not change with the scale is returned unchanged). Combined with readout mitigation, the linear extrapolation gives **0.9983** and the quadratic one 1.0000: almost all of the bias is gone. ZNE **without** the readout fix gives only **0.8291**, because folding does not amplify the readout error, so extrapolation cannot see it. The two methods remove different errors and work best together.

The weights also show the price. The quadratic weights are larger (1.875, −1.25, 0.375) than the linear ones, and a larger weight multiplies the shot noise of its measurement.

## Step 7: how certain is each result, and is it worth the shots?

**The bootstrap.** For the readout-mitigated value, a formula for σ is messy. The **bootstrap** measures it from the data: pretend the measured fractions are the truth, draw many new runs of the same number of shots from them (`rng.multinomial(shots, fractions)`), apply the same estimator to each, and take the standard deviation of the results. It works for any estimator, however complicated.

**Your task:** `bootstrap_sigma(counts, estimator, n_boot=200, seed=0)`:

1. `rng = np.random.default_rng(seed)`; the keys, their counts, the total number of shots, the fractions;
2. n_boot times: draw new counts with `rng.multinomial(shots, fractions)`, turn them into a dictionary, apply `estimator`;
3. return the standard deviation of the n_boot values, `np.std(values, ddof=1)`.

Do not change `counts`; build a new dictionary for every resample.

The cell after it takes a run of 4,000 shots at each scale (12,000 shots in all), gives every result with its σ, and compares each with the ideal value 1.

In [ ]:
def bootstrap_sigma(counts, estimator, n_boot=200, seed=0):
    """The standard deviation of estimator(resampled counts) over n_boot multinomial resamplings."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete bootstrap_sigma() first.")

In [ ]:
ro_est = lambda c: parity_expectation(mitigate_readout(c, READOUT_COURSE))
runs = {s: sample_counts(exact_s[s], SHOTS, seed=s) for s in SCALES}
E_raw = {s: parity_expectation(runs[s]) for s in SCALES}
E_ro = {s: ro_est(runs[s]) for s in SCALES}
sig_raw = {s: bootstrap_sigma(runs[s], parity_expectation) for s in SCALES}
sig_ro = {s: bootstrap_sigma(runs[s], ro_est) for s in SCALES}
print("Raw, scale 1: bootstrap sigma", round(sig_raw[1], 4), " formula expectation_sigma:", round(expectation_sigma(E_raw[1], SHOTS), 4))

results = {
    "raw (4,000 shots)": (E_raw[1], sig_raw[1]),
    "readout-mitigated (4,000 shots)": (E_ro[1], sig_ro[1]),
    "readout + ZNE linear (12,000 shots)": (zne(lin, E_ro), combined_sigma(lin, [sig_ro[s] for s in SCALES])),
    "readout + ZNE quadratic (12,000 shots)": (zne(quad, E_ro), combined_sigma(quad, [sig_ro[s] for s in SCALES])),
}
print(f"\n{'method':>40s}{'value':>9s}{'sigma':>9s}{'(1 - value) / sigma':>21s}")
for name, (v, sg) in results.items():
    print(f"{name:>40s}{v:9.4f}{sg:9.4f}{(1 - v) / sg:21.1f}")

**What to notice.** The bootstrap σ of the raw value agrees with the formula `expectation_sigma`, a good test of both. Each step of mitigation **removes bias and adds uncertainty**: with these runs the exact standard deviations are about 0.0093 (raw), 0.0111 (readout-mitigated), 0.0138 (with linear ZNE) and 0.0262 (with quadratic ZNE). The raw value is many σ below 1: its bias is significant in any run. The exact readout-mitigated value is about 2.1σ below 1 (bias 0.024, σ 0.011), so a single run of 4,000 shots may or may not show the remaining gate error: look at the last column of your run. Both ZNE results agree with 1 within their σ, the quadratic one with a much larger σ.

**The shot budget.** Which method is best depends on how many shots you can spend. A good measure is the **root-mean-square error**, RMS = √(bias² + σ²): the typical distance from the true value, counting both kinds of error. σ shrinks as 1/√shots; the bias does not. Run the cell: for a total budget of shots (split equally between the three scales for ZNE), it gives the RMS error of each method from the exact biases and σ.

In [ ]:
bias = {"raw": 1 - raw_s[1], "readout": 1 - ro_s[1], "readout + ZNE linear": 1 - zne(lin, ro_s),
        "readout + ZNE quadratic": 1 - zne(quad, ro_s)}
per_shot = {"raw": sig_raw[1] ** 2 * SHOTS, "readout": sig_ro[1] ** 2 * SHOTS,      # sigma^2 x shots does not depend on shots
            "readout + ZNE linear": combined_sigma(lin, [sig_ro[s] for s in SCALES]) ** 2 * 3 * SHOTS,
            "readout + ZNE quadratic": combined_sigma(quad, [sig_ro[s] for s in SCALES]) ** 2 * 3 * SHOTS}
budgets = [300, 1200, 3000, 12000, 48000, 300000, 3000000]
print(f"{'total shots':>12s}" + "".join(f"{m:>25s}" for m in bias))
for B in budgets:
    rms = {m: math.sqrt(bias[m] ** 2 + per_shot[m] / B) for m in bias}
    best = min(rms, key=rms.get)
    print(f"{B:12,d}" + "".join(f"{rms[m]:25.4f}" for m in bias) + "   best: " + best)

**What to notice.** With a few hundred shots, readout mitigation alone gives the smallest error: ZNE's larger σ costs more than the small bias it removes. From about **3,200 shots** in all, linear ZNE wins, because its σ falls below the remaining bias of readout mitigation. Quadratic ZNE removes the last bit of bias, but its σ is about twice the linear one, so it pays only when the total budget reaches roughly **2,000,000 shots**. (Your bootstrap σ values come from one run each, so your crossings may differ a little from these exact ones.) Mitigation is a trade: the more bias you remove, the more shots you need to keep the uncertainty below it.

## Step 8: more qubits

A larger GHZ state has more gates and more bits to misread. Run the cell: for 2 to 5 qubits it gives the exact raw, readout-mitigated and fully mitigated values, and the σ of each from a run of 4,000 shots per scale.

In [ ]:
print(f"{'qubits':>7s}{'raw':>16s}{'readout':>18s}{'readout + ZNE lin':>21s}")
table = {}
for n in range(2, 6):
    ex = {s: exact_probabilities(measured(fold_global(ghz_circuit(n), s))) for s in SCALES}
    rn = {s: sample_counts(ex[s], SHOTS, seed=10 * n + s) for s in SCALES}
    sr = {s: bootstrap_sigma(rn[s], ro_est) for s in SCALES}
    raw_n = parity_expectation(ex[1])
    ro_n = {s: ro_est(ex[s]) for s in SCALES}
    table[n] = (raw_n, ro_n[1], zne(lin, ro_n))
    print(f"{n:7d}{raw_n:9.4f} +- {expectation_sigma(raw_n, SHOTS):.3f}{ro_n[1]:11.4f} +- {sr[1]:.3f}"
          f"{zne(lin, ro_n):14.4f} +- {combined_sigma(lin, [sr[s] for s in SCALES]):.3f}")

**What to notice.** The raw value falls quickly with the number of qubits (0.873 for 2 qubits, 0.701 for 5), and so does the readout-mitigated one (0.987 to 0.955). Full mitigation stays close to 1 (0.9995 to 0.9939), but its σ grows with n: more bits to correct means a larger amplification of the shot noise. For a much larger circuit the mitigated value would need many more shots to be as certain, which is the main limit of these methods.

## Step 9: your personal check

Open your capstone quiz in Canvas. Question 1 shows your own numbers: NQ (2 to 5 qubits), P2 and READOUT (0.005 to 0.030). Type them below and run the next two cells. The check cell tests the three statistics functions of Step 1 and every function you wrote for this capstone. Only if every test passes does it print your **verification value**: 10,000 times the gain of full mitigation for your noise model, (readout + linear ZNE at scales 1, 3, 5) minus the raw value, both exact, for an NQ-qubit GHZ state, rounded to a whole number. For example, NQ, P2, READOUT = 3, 0.01, 0.02 gives 1874.

In [ ]:
NQ = 0    # your number from Canvas
P2 = 0    # your number from Canvas
READOUT = 0    # your number from Canvas

In [ ]:
# The course's check code is in l3_m6_common.py and l3_m6_mit_check.py, next to this notebook.
# It is kept out of the notebook so that the course's reference solutions are not on screen.
# You may open the file to see how your functions are tested, but write your own functions first.
from l3_m6_mit_check import check_l3_m6_mit

passed, messages, value = check_l3_m6_mit(fraction_sigma, expectation_sigma, combined_sigma, ghz_circuit, parity_expectation,
                                          mitigate_readout, fold_global, zne_coefficients, bootstrap_sigma,
                                          NQ, P2, READOUT)
for m in messages:
    print(m)
if passed:
    print("\nAll tests passed. Your verification value is", value)
else:
    print("\nNot yet: fix the item above and run this cell again.")

## Limits of this experiment

- **One expectation value, one state.** ⟨X X X⟩ of a GHZ state is a single number whose ideal value you know. In a real application the ideal value is unknown, so you cannot see the remaining bias; you can only check that methods agree within their σ.
- **The noise model is known exactly.** Readout mitigation used the exact READOUT. On a device the confusion matrix is itself measured with shots (calibration circuits) and drifts over time, which adds uncertainty the bootstrap here does not include.
- **Folding assumes the noise scales.** Global folding triples the gate noise only if U⁻¹ U makes the same errors as U. On hardware, the compiler may cancel U⁻¹ U, coherent errors can cancel or add, and the noise is not exactly depolarizing, so the extrapolation can be biased.
- **The extrapolation model is a guess.** A line or a parabola is a choice. The values here fall almost exactly as a smooth curve, which is why both work; real data may not.
- **Larger circuits.** σ grows quickly with the number of qubits and the depth. These methods suit small circuits and a modest number of measured values.

These limits are not failures of your experiment: every experiment has them. Saying what a result does and does not show is part of reporting it.

## Optional: run it on a real IBM quantum computer

If you have an IBM Quantum account with an Open Plan instance (see **Set Up Your Tools** in Start Here), the [Module 6 hardware notebook in Colab](https://colab.research.google.com/github/CSU-Physics/quantum-computing-notebooks/blob/main/colab/level3/QC-L3-M6-hardware-colab.ipynb) runs this capstone's reference circuits on a real computer and compares them with the ideal values and with this noise model. Your API key stays in Colab's **Secrets**; never type it into a cell. The Open Plan gives up to 10 minutes of quantum computer time every 28 days; the hardware notebook uses well under a minute of it, but the queue can take from seconds to hours. This step is not graded, and the badge does not depend on it.

## Your capstone summary (required, not graded)

Write five short answers here (double-click this cell to edit it). They are how a result is reported in a lab notebook or a paper, and they are good preparation for the capstone quiz. Then paste them into **Your Capstone Summary** in the Course Completion module in Canvas: it is not graded, but it is required for the badge.

1. **Question.** What did you want to find out?
2. **Method.** Which circuits, which noise model (P2, READOUT), how many shots, which mitigation or decoding?
3. **Result.** Your main numbers, each with its uncertainty (for example 0.976 ± 0.010).
4. **Interpretation.** What do the numbers say about the question? Which differences are significant (more than 3σ)?
5. **Limits.** What does this experiment not show?

**Next in Canvas:** the **Capstone Quiz — Mitigation Study**, then the Module 6 time log.